# E-Commerce Customer Segmentation using RFM, K-Means, DBSCAN and PCA

**Goal:** Segment customers using Recency, Frequency, and Monetary (RFM) features, compare K-Means and DBSCAN, and visualize clusters with PCA.

This notebook includes the small dataset directly, so you can run it in Google Colab without uploading an Excel file. It also creates an Excel copy of the dataset.

## 1. Import required libraries

Run this cell first. These libraries are available in Google Colab in most cases.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print("Libraries imported successfully!")

## 2. Create the mini RFM dataset

- **Recency:** Days since the customer's last purchase (smaller is more recent).
- **Frequency:** Number of orders (larger means more orders).
- **Monetary:** Total amount spent by the customer.

The dataset is illustrative/synthetic and is included directly in this notebook.

In [ ]:
data = {
    "CustomerID": ["C001","C002","C003","C004","C005",
                   "C006","C007","C008","C009","C010",
                   "C011","C012","C013","C014","C015"],
    "Recency":   [3, 5, 7, 10, 12, 15, 20, 25, 30, 35, 45, 60, 75, 100, 150],
    "Frequency": [15,12,14,10,9,8,6,5,4,3,3,2,2,1,1],
    "Monetary":  [25000,22000,24000,18000,16000,14000,10000,8000,6000,4500,4000,2500,2000,1000,50000]
}

df = pd.DataFrame(data)
display(df)
print("Dataset shape:", df.shape)

## 3. Save the dataset as an Excel file (optional)

Running this cell creates `ecommerce_customer_rfm_dataset.xlsx`. In Google Colab, the download prompt will appear.

In [ ]:
excel_file = "ecommerce_customer_rfm_dataset.xlsx"
df.to_excel(excel_file, index=False)

try:
    from google.colab import files
    files.download(excel_file)
except ImportError:
    print(f"Excel file saved as {excel_file}")

## 4. Explore the data

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nRFM descriptive statistics:")
display(df[["Recency", "Frequency", "Monetary"]].describe())

## 5. Standardize the RFM features

Scaling is important because Monetary values are much larger than Recency and Frequency. We use `StandardScaler` so each feature is on a comparable scale.

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized data shape:", X_scaled.shape)

## 6. Choose a K for K-Means: elbow and silhouette methods

- **Elbow method:** Looks for a bend in the within-cluster sum of squares (inertia).
- **Silhouette score:** Higher is generally better; it ranges from -1 to 1.

In [ ]:
inertias = []
for k in range(1, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 7), inertias, marker="o")
plt.title("K-Means Elbow Method")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(range(1, 7))
plt.grid(True)
plt.show()

silhouette_scores = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, labels)

print("K-Means silhouette scores:")
for k, score in silhouette_scores.items():
    print(f"K={k}: {score:.3f}")

## 7. Apply K-Means clustering

For this mini case study, we use **K = 3** to create three broad customer groups. Cluster numbers are arbitrary; interpret each group using its RFM averages.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)

print("Customer assignments:")
display(df[["CustomerID", "Recency", "Frequency", "Monetary", "KMeans_Cluster"]])

print("\nAverage RFM values by K-Means cluster:")
kmeans_summary = df.groupby("KMeans_Cluster")[features].mean().round(2)
display(kmeans_summary)

print("Customers per K-Means cluster:")
display(df["KMeans_Cluster"].value_counts().sort_index())

## 8. Apply DBSCAN clustering

DBSCAN groups nearby points and marks points in sparse regions as **noise** (`-1`). Its output depends on the `eps` and `min_samples` settings. Because this is a very small dataset, treat the result as a demonstration rather than a universal customer segmentation.

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=3)
df["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

print("Customer assignments:")
display(df[["CustomerID", "Recency", "Frequency", "Monetary", "DBSCAN_Cluster"]])

print("DBSCAN cluster counts (-1 means noise):")
display(df["DBSCAN_Cluster"].value_counts().sort_index())

print("\nAverage RFM values by DBSCAN cluster (excluding noise):")
non_noise = df[df["DBSCAN_Cluster"] != -1]
if non_noise["DBSCAN_Cluster"].nunique() > 0:
    display(non_noise.groupby("DBSCAN_Cluster")[features].mean().round(2))
else:
    print("DBSCAN found no non-noise clusters with the current settings. Try adjusting eps or min_samples.")

## 9. Reduce the RFM data to two dimensions using PCA

PCA creates two new features (PC1 and PC2) that retain as much variation as possible for a two-dimensional view. The clustering itself was done using the standardized original RFM features.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]

print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("Total variance retained by 2 components:", round(pca.explained_variance_ratio_.sum(), 3))

## 10. Visualize K-Means clusters using PCA

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(df["PC1"], df["PC2"], c=df["KMeans_Cluster"], cmap="viridis", s=90)
for _, row in df.iterrows():
    plt.annotate(row["CustomerID"], (row["PC1"], row["PC2"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
plt.title("K-Means Customer Segments (PCA view)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="K-Means cluster")
plt.grid(True)
plt.show()

## 11. Visualize DBSCAN clusters using PCA

Noise points (`-1`) are shown as their own label in the color scale.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(df["PC1"], df["PC2"], c=df["DBSCAN_Cluster"], cmap="plasma", s=90)
for _, row in df.iterrows():
    plt.annotate(row["CustomerID"], (row["PC1"], row["PC2"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
plt.title("DBSCAN Customer Segments (PCA view)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="DBSCAN cluster (-1 = noise)")
plt.grid(True)
plt.show()

## 12. Compare clustering quality

K-Means silhouette is calculated across all customers. For DBSCAN, we calculate it only for non-noise customers and only if at least two clusters remain. These scores are not perfectly comparable because the DBSCAN score excludes noise points.

In [ ]:
kmeans_sil = silhouette_score(X_scaled, df["KMeans_Cluster"])
print(f"K-Means silhouette score (K=3): {kmeans_sil:.3f}")

dbscan_mask = df["DBSCAN_Cluster"] != -1
dbscan_labels = df.loc[dbscan_mask, "DBSCAN_Cluster"]
if dbscan_mask.sum() >= 3 and dbscan_labels.nunique() >= 2 and dbscan_labels.nunique() < dbscan_mask.sum():
    dbscan_sil = silhouette_score(X_scaled[dbscan_mask.to_numpy()], dbscan_labels)
    print(f"DBSCAN silhouette score (noise excluded): {dbscan_sil:.3f}")
else:
    print("DBSCAN silhouette score is not defined for this result (needs at least 2 non-noise clusters).")

print("\nFinal dataset with cluster labels and PCA coordinates:")
display(df)

## 13. Case study interpretation and conclusion

Use the printed cluster averages and customer assignments above to interpret the groups; do not assume that cluster number 0, 1, or 2 has a fixed meaning.

Typical business interpretation:
- **Low Recency + high Frequency + high Monetary:** recent, frequent, high-value customers; consider loyalty rewards and VIP offers.
- **High Recency + low Frequency + low Monetary:** inactive or at-risk customers; consider a win-back campaign.
- **High Monetary but high Recency / low Frequency:** potentially a high-spend customer who has not returned; consider personalized re-engagement.
- **DBSCAN noise (`-1`):** customers whose standardized RFM profiles are not in a sufficiently dense group under the chosen settings; investigate them individually.

**Conclusion:** K-Means provides a fixed number of customer segments, while DBSCAN can identify dense groups and flag sparse observations as noise. PCA makes the three-dimensional RFM profiles easier to visualize in two dimensions. Since this dataset has only 15 illustrative customers, use a larger real retail dataset for reliable business decisions.